In [29]:
from minio import Minio
from dotenv import load_dotenv
import os
import geopandas as gpd
import requests
from datetime import datetime
from shapely.geometry import Polygon
from pathlib import Path
import hashlib
from tqdm import tqdm

load_dotenv()

True

In [10]:
client = Minio(
	endpoint=os.environ['S3_ENDPOINT'],
	access_key=os.environ['ACCESS_KEY_ID'],
	secret_key=os.environ['SECRET_ACCESS_KEY'],
	secure=True,
)

In [11]:
def retrieve_model(name):
    response = requests.get("https://api.eotdl.com/models?name=" + name)
    return response.json()

In [12]:
model = "MassachusettsRoadsS2Model"
model_id = retrieve_model(model)["id"]
model_id


'683eece500cf9ffafe807823'

In [25]:
# get files from bucket

objects = list(client.list_objects(os.environ['BUCKET'], prefix=model_id, recursive=True))

for obj in objects:
    print(obj.object_name)
    if not obj.object_name.endswith('/'):
        client.fget_object(
            os.environ['BUCKET'],
            obj.object_name,
            'data/' + obj.object_name.split('/')[-1],
        )


683eece500cf9ffafe807823/
683eece500cf9ffafe807823/README.md
683eece500cf9ffafe807823/catalog.v1.parquet
683eece500cf9ffafe807823/model.onnx


In [26]:
gdf = gpd.read_parquet("data/catalog.v1.parquet")
gdf

,type,stac_version,stac_extensions,datetime,id,bbox,geometry,assets,links,repository
0,Feature,1.0.0,[],2025-06-03 14:39:00.994848,README.md,"{'xmax': 0.0, 'xmin': 0.0, 'ymax': 0.0, 'ymin'...",POLYGON EMPTY,{'asset': {'checksum': '214490dad39642b6013abf...,[],eotdl
1,Feature,1.0.0,[],2025-06-03 14:39:00.995097,model.onnx,"{'xmax': 0.0, 'xmin': 0.0, 'ymax': 0.0, 'ymin'...",POLYGON EMPTY,{'asset': {'checksum': 'f313a4bf02afd1a6b55829...,[],eotdl


In [39]:
def calculate_checksum(file_path):
    sha1_hash = hashlib.sha1()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            sha1_hash.update(chunk)
    return sha1_hash.hexdigest()

def create_stac_item(item_id, asset_href):
    metadata = {
        'type': 'Feature',
        'stac_version': '1.0.0',
        'stac_extensions': [],
        'datetime': datetime.now(),  # must be native timestamp (https://github.com/apache/parquet-format/blob/master/LogicalTypes.md#timestamp)
        'id': item_id,
        'bbox': {
            'xmin': 0.0,
            'ymin': 0.0,
            'xmax': 0.0,
            'ymax': 0.0
        }, 
        'geometry': Polygon(), # empty polygon
        'assets': { 'asset': { 
            'href': asset_href,
            'checksum': calculate_checksum(asset_href),
            'timestamp': datetime.now(),
            'size': Path(asset_href).stat().st_size,
        }},
        "links": [],
        # anything below are properties (need at least one!)
        'repository': 'eotdl',			
    }
    if item_id == 'model.onnx':
        model_metadata = {
            "mlm:name": "model.onnx", # name of the asset ? otherwise, how can we know which asset to use ?
            "mlm:framework": "ONNX",  # only framework support for now
            "mlm:architecture": "U-Net",
            "mlm:tasks": ["segmentation"], # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#task-enum
            "mlm:input": { # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#model-input-object
                "name": "RGB statellite image (Sentinel 2)",
                "bands": [
                    "red",
                    "green",
                    "blue"
                ],
                "input": { # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#input-structure-object
                    "shape": [
                        -1,
                        3,
                        -1, # should be divisble by 16
                        -1 # should be divisble by 16
                    ],
                    "dim_order": [
                        "batch",
                        "channel",
                        "height",
                        "width"
                    ],
                    "data_type": "float32",
                    # we should add here the resize to nearest divisible by 16
                    # "pre_processing_function": { # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#processing-expression
                    # 	"format": 
                    # 	"expression": 
                    # }
                    "description": "Model trained with Sentinel2 RGB images, can work with any dimensions as long as they are divisible by 16"
                }
            },
            "mlm:output": {
                "name": "road binary mask",
                "tasks": ["segmentation"], # redundant ?
                "result": { # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#result-structure-object
                    "shape": [-1, 1, -1, -1],
                    "dim_order": [
                        "batch",
                        "channel",
                        "height",
                        "width"
                    ],
                    "data_type": "uint8",
                    "description": "Binary mask of the road segmentation. 1 for road, 0 for background",
                    # "post_processing_function": { # https://github.com/crim-ca/mlm-extension?tab=readme-ov-file#processing-expression
                    # }
                },
            }
        }
        metadata.update(model_metadata)
    return metadata

In [40]:
catalog_path = Path("data/catalog.parquet")
files = ['model.onnx', 'README.md']
data = []
for file in tqdm(files):
    file_path = 'data' / Path(file)
    if file_path.is_file():
        relative_path = os.path.relpath(file_path, catalog_path.parent)
        absolute_path = str(file_path)
        stac_item = create_stac_item(relative_path, absolute_path)
        stac_item['assets']['asset']['href'] = f"https://api.eotdl.com/datasets/{model_id}/stage/{relative_path}"
        data.append(stac_item)
gdf = gpd.GeoDataFrame(data, geometry='geometry')
gdf.to_parquet(catalog_path)

100%|██████████| 2/2 [00:00<00:00, 25.38it/s]


In [41]:
gdf = gpd.read_parquet("data/catalog.parquet")
gdf

,type,stac_version,stac_extensions,datetime,id,bbox,geometry,assets,links,repository,mlm:name,mlm:framework,mlm:architecture,mlm:tasks,mlm:input,mlm:output
0,Feature,1.0.0,[],2025-09-23 11:42:09.825314,model.onnx,"{'xmax': 0.0, 'xmin': 0.0, 'ymax': 0.0, 'ymin'...",POLYGON EMPTY,{'asset': {'checksum': 'f313a4bf02afd1a6b55829...,[],eotdl,model.onnx,ONNX,U-Net,[segmentation],"{'bands': ['red', 'green', 'blue'], 'input': {...","{'name': 'road binary mask', 'result': {'data_..."
1,Feature,1.0.0,[],2025-09-23 11:42:09.903335,README.md,"{'xmax': 0.0, 'xmin': 0.0, 'ymax': 0.0, 'ymin'...",POLYGON EMPTY,{'asset': {'checksum': '214490dad39642b6013abf...,[],eotdl,None,None,None,None,None,None


In [42]:
client.fput_object(
    os.environ['BUCKET'],
    f'{model_id}/catalog.v1.parquet',
    catalog_path,
)   